In [1]:
import csv

def fix_csv(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)
        for row in reader:
            # Attempt to fix potential quote issues:
            fixed_row = [field.replace('"', '""') for field in row]  # Escape inner quotes
            writer.writerow(fixed_row)

# Usage:
fix_csv('/home/weichao/Sentiment Analysis/data/Combined Data.csv', '/home/weichao/Sentiment Analysis/data/fixed_data.csv')  # Creates a new fixed CSV file
CSV_FILE_PATH = '/home/weichao/Sentiment Analysis/data/fixed_data.csv'  # Use the fixed CSV for loading

In [2]:
import pandas as pd
import tensorflow as tf
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split 
from sklearn.utils import shuffle 

# --- Configuration ---
TEXT_COLUMN = 'statement' # Name of the column containing the text
LABEL_COLUMN = 'status' # Name of the column containing the numerical labels

# Choose a pre-trained model tokenizer.
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased'

MAX_LENGTH = 128  # Maximum sequence length for tokenization
BATCH_SIZE = 16   # Batch size for training

TRAIN_VAL_SIZE = 50000 # Number of samples for combined training and validation
RANDOM_STATE = 42      # Seed for reproducibility during shuffling

# --- 1. Load Data using Pandas ---
try:
    df = pd.read_csv(CSV_FILE_PATH, on_bad_lines='skip')
    print(f"Successfully loaded {len(df)} rows from {CSV_FILE_PATH}")
    # Basic validation
    if TEXT_COLUMN not in df.columns:
        raise ValueError(f"Text column '{TEXT_COLUMN}' not found in the CSV.")
    if LABEL_COLUMN not in df.columns:
        raise ValueError(f"Label column '{LABEL_COLUMN}' not found in the CSV.")
    if len(df) <= TRAIN_VAL_SIZE:
        raise ValueError(f"Dataset size less than ({TRAIN_VAL_SIZE}). Please provide a larger dataset.")

except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_FILE_PATH}")
except ValueError as ve:
    print(f"Error: {ve}")
except Exception as e:
    print(f"An unexpected error occurred while loading the CSV: {e}")
    
# --- 2. Shuffle the Entire Dataset ---
df_shuffled = shuffle(df, random_state=RANDOM_STATE)

label_encoder = LabelEncoder()
df_shuffled_labels = label_encoder.fit_transform(df_shuffled[LABEL_COLUMN])
df_shuffled_texts = [str(text) for text in df_shuffled[TEXT_COLUMN].tolist()]

# Take the first TRAIN_VAL_SIZE rows for training and validation
train_shuffled_text_df = df_shuffled_texts[:TRAIN_VAL_SIZE].copy()
train_shuffled_labels_df = df_shuffled_labels[:TRAIN_VAL_SIZE].copy()

# Take the remaining rows for testing
test_shuffled_text_df = df_shuffled_texts[TRAIN_VAL_SIZE:].copy()
test_shuffled_labels_df = df_shuffled_labels[TRAIN_VAL_SIZE:].copy()

print("\n--- Data Loading and Splitting Complete ---")
# Translate label to int


print(f"Sample Text: {test_shuffled_text_df[0]}")
print(f"Sample Label: {test_shuffled_labels_df[0]}")
print(f"Number of unique labels: {len(set(train_shuffled_labels_df))}")


# --- 2. Load Tokenizer ---
print(f"\nLoading tokenizer for model: {PRETRAINED_MODEL_NAME}...")
try:
    # AutoTokenizer automatically selects the correct tokenizer class
    # based on the pre-trained model name.
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading tokenizer: {e}")
    print("Please ensure the model name is correct and you have an internet connection.")

# --- 3. Tokenize Data ---
print(f"\nTokenizing text data (max length: {MAX_LENGTH})...")

# Use the tokenizer directly on the list of texts.
# padding='max_length': Pad shorter sequences to MAX_LENGTH.
# truncation=True: Truncate longer sequences to MAX_LENGTH.
# return_tensors='tf': Return TensorFlow tensors.
try:
    # Tokenize in batches for potentially large datasets (though for moderate sizes,
    # tokenizing all at once might be faster if memory allows)
    # Here we tokenize all at once for simplicity
    encodings = tokenizer(
        train_shuffled_text_df,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='tf' # Return TensorFlow tensors directly
    )
    print("Tokenization complete.")
    print("Keys in encoding:", encodings.keys())
    print("Shape of input_ids:", encodings['input_ids'].shape)

except Exception as e:
    print(f"An error occurred during tokenization: {e}")
    exit()


# --- 4. Create TensorFlow Dataset ---
print("\nCreating TensorFlow dataset...")

# Create a dataset from the tokenized encodings (which is a dictionary of tensors) and the labels.
# tf.data.Dataset.from_tensor_slices works well when data fits in memory.
try:
    # Create dataset from the dictionary of encodings and labels
    tf_dataset = tf.data.Dataset.from_tensor_slices(
        (
            # The input data for the model is typically a dictionary
            # containing 'input_ids', 'attention_mask', etc.
            dict(encodings),
            # The corresponding labels
            train_shuffled_labels_df
        )
    )
    print("TensorFlow dataset created successfully.")

    # Optional: Shuffle and Batch the dataset
    # Buffer size should ideally be larger than the dataset size for perfect shuffling,
    # but a large buffer works well in practice.
    buffer_size = len(df)
    tf_dataset = tf_dataset.shuffle(buffer_size=buffer_size).batch(BATCH_SIZE)
    print(f"Dataset shuffled and batched (batch size: {BATCH_SIZE}).")

    # Optional: Prefetch for performance
    tf_dataset = tf_dataset.prefetch(tf.data.AUTOTUNE)
    print("Dataset prefetching enabled.")

    # You can inspect an element
    for batch in tf_dataset.take(1):
        input_batch, label_batch = batch
        print("\nSample batch structure:")
        print("Input keys:", input_batch.keys())
        print("Shape of input_ids in batch:", input_batch['input_ids'].shape)
        print("Shape of labels in batch:", label_batch.shape)
        print("Sample labels in batch:", label_batch.numpy())

except Exception as e:
    print(f"An error occurred while creating the TensorFlow dataset: {e}")
    exit()

print("\n--- Data Loading and Preparation Complete ---")
print("The variable 'tf_dataset' now holds your prepared data.")

2025-05-02 19:30:41.077061: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 19:30:41.087412: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746178241.099324   30522 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746178241.102804   30522 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746178241.112143   30522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Successfully loaded 53043 rows from /home/weichao/Sentiment Analysis/data/fixed_data.csv

--- Data Loading and Splitting Complete ---
Sample Text: well, that's okay, as long as it helps him relax and think more clearly.
Sample Label: 3
Number of unique labels: 7

Loading tokenizer for model: distilbert-base-uncased...
Tokenizer loaded successfully.

Tokenizing text data (max length: 128)...
Tokenization complete.
Keys in encoding: dict_keys(['input_ids', 'attention_mask'])
Shape of input_ids: (50000, 128)

Creating TensorFlow dataset...
TensorFlow dataset created successfully.
Dataset shuffled and batched (batch size: 16).
Dataset prefetching enabled.

Sample batch structure:
Input keys: dict_keys(['input_ids', 'attention_mask'])
Shape of input_ids in batch: (16, 128)
Shape of labels in batch: (16,)
Sample labels in batch: [2 6 4 6 3 3 2 3 6 3 5 2 3 2 6 6]

--- Data Loading and Preparation Complete ---
The variable 'tf_dataset' now holds your prepared data.


W0000 00:00:1746178247.068109   30522 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-05-02 19:30:47.305963: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification
import numpy as np 

# Check if GPU is available and set it as the default device
if tf.config.list_physical_devices('GPU'):
    print("GPU is available. Using GPU for computations.")
    tf.debugging.set_log_device_placement(True)  # Optional: Logs device placement for debugging
else:
    print("GPU is not available. Using CPU for computations.")

PRETRAINED_MODEL_NAME = 'distilbert-base-uncased' # Placeholder - CHANGE IF NEEDED

try:
    # Assuming 'labels' list exists from the previous data loading step
    if 'labels' in globals() and train_shuffled_labels_df is not None:
         NUM_LABELS = len(np.unique(train_shuffled_labels_df))
         print(f"Inferred number of labels from data: {NUM_LABELS}")
    else:
         # *** SET MANUALLY IF 'labels' is not available ***
         NUM_LABELS = 7 # Example: Replace with your actual number of classes
         print(f"Number of labels set manually: {NUM_LABELS}")
except NameError:
     # *** SET MANUALLY IF 'labels' is not defined ***
     NUM_LABELS = 7 # Example: Replace with your actual number of classes
     print(f"Number of labels set manually (due to NameError): {NUM_LABELS}")
except Exception as e:
    print(f"Could not automatically determine number of labels: {e}")
    # *** SET MANUALLY AS FALLBACK ***
    NUM_LABELS = 7 # Example: Replace with your actual number of classes
    print(f"Number of labels set manually (due to exception): {NUM_LABELS}")


# --- Load Pre-trained Model ---
print(f"\nLoading pre-trained model: {PRETRAINED_MODEL_NAME} for sequence classification...")

try:
    # TFAutoModelForSequenceClassification automatically loads the TensorFlow
    # version of the model with a classification head.
    # - PRETRAINED_MODEL_NAME: Specifies the model architecture and weights.
    # - num_labels: Configures the output layer of the classification head.
    model = TFAutoModelForSequenceClassification.from_pretrained(
        PRETRAINED_MODEL_NAME,
        num_labels=NUM_LABELS
    )

    print("\nModel loaded successfully!")

    # You can print the model summary to see the architecture
    print("\nModel Summary:")
    model.summary()

    # Store the model configuration if needed later
    model_config = model.config
    print(f"\nModel Config Loaded (Example: id2label mapping): {model_config.id2label}")


except OSError as e:
    print(f"\nError loading model: {e}")
    print(f"Could not find model '{PRETRAINED_MODEL_NAME}' on Hugging Face Hub or locally.")
    print("Please ensure the model name is correct, you have an internet connection,")
    print("and the model is compatible with TFAutoModelForSequenceClassification.")
    # Optionally exit if the model is critical
    # exit()
except Exception as e:
    print(f"\nAn unexpected error occurred while loading the model: {e}")
    # Optionally exit
    # exit()

print("\n--- Model Loading Complete ---")
# The variable 'model' now holds your TensorFlow/Keras model, ready for compilation and fine-tuning.
import tensorflow as tf
import numpy as np
import math


NUM_LABELS = 7  # As specified by the user
LEARNING_RATE = 5e-5  # Common learning rate for fine-tuning transformers (can be tuned: 2e-5, 3e-5, 5e-5)

EPOCHS = 3

print("Preparing training and validation datasets...")

try:
    # Full run: Split the dataset (e.g., 90% train, 10% val)
    dataset_size = tf.data.experimental.cardinality(tf_dataset).numpy()
    if dataset_size == tf.data.experimental.UNKNOWN_CARDINALITY or dataset_size <= 0:
        print("Warning: Could not determine dataset size automatically. Using estimated sizes or default split.")
        # Fallback strategy if size is unknown - this might be inaccurate
        # Consider calculating size beforehand if possible
        estimated_total_batches = 1000 # Provide a reasonable estimate or calculate if possible
        val_size = int(0.15 * estimated_total_batches)
        train_size = estimated_total_batches - val_size
        print(f"Using estimated split: ~{train_size} train batches, {val_size} val batches.")
        # Apply the split using take/skip based on estimates
        val_dataset = tf_dataset.take(val_size)
        train_dataset = tf_dataset.skip(val_size)
    else:
        print(f"Total batches: {dataset_size}")
        val_size = max(1, int(0.15 * dataset_size))  # Ensure at least 1 batch for validation
        train_size = dataset_size - val_size
        print(f"Using {train_size} batches for training and {val_size} batches for validation (Full Run).")
        # Apply the split
        val_dataset = tf_dataset.take(val_size)
        train_dataset = tf_dataset.skip(val_size)

    # Verify dataset sizes (optional)
    print(f"Train dataset cardinality: {tf.data.experimental.cardinality(train_dataset)}")
    print(f"Validation dataset cardinality: {tf.data.experimental.cardinality(val_dataset)}")
    print("Dataset preparation complete.")


except NameError as ne:
    print(f"Error: Required variable not found: {ne}")
    print("Ensure 'tf_dataset' exists from the data loading step.")
    exit()
except Exception as e:
    print(f"An error occurred during dataset splitting: {e}")
    exit()


# --- 2. Compile the Model ---
print("\nCompiling the model...")

# Use AdamW optimizer, which is recommended for Transformers
optimizer = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE)

# Use SparseCategoricalCrossentropy loss because labels are integers (0-6)
# Set from_logits=True because the Hugging Face model outputs raw scores (logits)
# before the final activation function (like softmax).
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Use SparseCategoricalAccuracy metric for monitoring
metric = tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')

try:
    # Ensure the loaded model has the correct number of labels in its config
    if model.config.num_labels != NUM_LABELS:
        print(f"Warning: Model config has {model.config.num_labels} labels, but NUM_LABELS is set to {NUM_LABELS}.")
        print("Re-check model loading step or NUM_LABELS setting.")
        # Potentially exit or try to proceed cautiously
        # exit()

    model.compile(optimizer=optimizer, loss=loss_function, metrics=[metric])
    print("Model compiled successfully.")

except NameError as ne:
    print(f"Error: Required variable not found: {ne}")
    print("Ensure 'model' exists from the model loading step.")
    exit()
except Exception as e:
    print(f"An error occurred during model compilation: {e}")
    exit()


# --- 3. Fine-tune the Model ---
print(f"\nStarting fine-tuning for {EPOCHS} epochs...")

# Optional: Add callbacks for better training control
# tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
# early_stopping_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=1) # Stop if val_loss doesn't improve for 1 epoch

try:
    # Check if datasets are empty before fitting
    if tf.data.experimental.cardinality(train_dataset) == 0:
         print("Error: Training dataset is empty. Check dataset splitting logic and source data.")
         exit()
    if tf.data.experimental.cardinality(val_dataset) == 0:
         print("Error: Validation dataset is empty. Check dataset splitting logic and source data.")
         exit()

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS
        # Optional: Add callbacks here
        # callbacks=[tensorboard_callback, early_stopping_callback]
    )

    print("\nFine-tuning complete!")
    print("Training History:", history.history)

except Exception as e:
    print(f"\nAn error occurred during model training: {e}")
    # Common issues: OOM (Out of Memory) errors - try reducing BATCH_SIZE.
    # Input data format errors - double-check the tf_dataset structure.

print("\n--- Fine-tuning Process Finished ---")



GPU is not available. Using CPU for computations.
Number of labels set manually: 7

Loading pre-trained model: distilbert-base-uncased for sequence classification...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 


Model loaded successfully!

Model Summary:
Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  5383      
                                                                 
 dropout_19 (Dropout)        multiple                  0 (unused)
                                                                 
Total params: 66958855 (255.43 MB)
Trainable params: 66958855 (255.43 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________

M

KeyboardInterrupt: 

In [8]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))
if gpus:
  print("GPU Found!")
  for gpu in gpus: print("Details:", gpu)
else:
    print("GPU not found.")

TensorFlow Version: 2.19.0
Num GPUs Available:  0
GPU not found.
